# HW3_2024

## 1
### A

In [ ]:
import numpy as np
from sympy import symbols, simplify, expand

# Given data points
data_points = np.array([
    [-2, 27.97490802],
    [0, 2.09014286],
    [2, 16.04639879]
])

x_values = data_points[:, 0]
y_values = data_points[:, 1]

# 1. Lagrange interpolation polynomial
x = symbols('x')

def lagrange_interpolation(x_vals, y_vals):
    n = len(x_vals)
    lagrange_poly = 0
    for i in range(n):
        # Compute basis polynomial L_i(x)
        L_i = 1
        for j in range(n):
            if i != j:
                L_i *= (x - x_vals[j]) / (x_vals[i] - x_vals[j])
        lagrange_poly += L_i * y_vals[i]
    return simplify(expand(lagrange_poly))

P_x = lagrange_interpolation(x_values, y_values)

# 2. Least squares fitting (second-degree polynomial)
# Fit y = ax^2 + bx + c
X_matrix = np.vstack([x_values**2, x_values, np.ones_like(x_values)]).T
coefficients, _, _, _ = np.linalg.lstsq(X_matrix, y_values, rcond=None)

# Construct polynomial Q(x) = ax^2 + bx + c
a, b, c = coefficients
Q_x = simplify(expand(a * x**2 + b * x + c))

P_x, Q_x

从结果上看，这两种方法的多项式几乎相同，这是因为给定的数据点符合一个近似的二次曲线，并且随机噪声较小（$|\epsilon(x)| < 0.1$）。

#### 比较分析
- **插值法**：
  - **优点**：多项式严格通过所有数据点，适合数据点较少、噪声较小的场景。
  - **缺点**：对数据中的噪声敏感，尤其是当数据点数较多时，插值多项式可能出现不稳定的震荡现象（Runge现象）。

- **拟合法**：
  - **优点**：通过平滑数据来减小噪声对结果的影响，尤其适合处理含随机噪声的数据。
  - **缺点**：拟合的多项式可能无法严格通过采样点。

### B

In [ ]:
import numpy as np

# 定义函数 g(x) 和其导数 g'(x)
def g(x):
    return x**3 - 2*x**2 + x

def g_prime_exact(x):
    return 3*x**2 - 4*x + 1

# 前向差分公式计算 D(h)
def forward_difference(g, x, h):
    return (g(x + h) - g(x)) / h

# x 的值和步长 h
x_val = 1.0
h1 = 0.1
h2 = 0.05

# 计算前向差分
D_h1 = forward_difference(g, x_val, h1)
D_h2 = forward_difference(g, x_val, h2)

# 使用 Richardson 外推法提高精度
# 二阶精度 (O(h^2))
D_O_h2 = D_h2 + (D_h2 - D_h1) / (2**2 - 1)

# 使用 Richardson 外推法提高到 O(h^4) 精度
# 四阶精度 (O(h^4))
D_O_h4 = D_O_h2 + (D_O_h2 - D_h2) / (2**2 - 1)


# 准确导数值
g_prime_at_1 = g_prime_exact(x_val)

# 计算误差
error_D_h1 = abs(D_h1 - g_prime_at_1)
error_D_h2 = abs(D_h2 - g_prime_at_1)
error_D_O_h2 = abs(D_O_h2 - g_prime_at_1)
error_D_O_h4 = abs(D_O_h4 - g_prime_at_1)

# 输出结果
D_h1, D_h2, D_O_h2, D_O_h4, g_prime_at_1, error_D_h1, error_D_h2, error_D_O_h2, error_D_O_h4


1. 前向差分结果：
   - $ D(h_1 = 0.1) = 0.11 $
   - $ D(h_2 = 0.05) = 0.0525 $

2. 使用 Richardson 外推法提高到二阶精度的结果：
   - $ D_{\text{精确}} (O(h^2)) = 0.0333 $

3. 使用 Richardson 外推法提高到四阶精度 $ O(h^4) $ 的结果：  
   $ D_{\text{精确}} (O(h^4)) = 0.02694 $

四阶精度进一步降低了误差，相比二阶精度的结果有了更高的准确性。

4. 准确导数值：
   - $ g'(1) = 0 $

5. 误差分析：
   - $ D(h_1) $ 的误差：$ 0.11 $
   - $ D(h_2) $ 的误差：$ 0.0525 $
   - $ D_{\text{精确}} (O(h^2)) $ 的误差：$ 0.0333 $
   - $ D_{\text{精确}} (O(h^4)) $ 的误差：$ 0.02694 $

Richardson 外推法成功降低了误差，提高了计算结果的精度。

### C
### 方法 1: 复合梯形公式

在复合梯形公式中，我们将积分区间 $[0, \pi]$ 划分为两个子区间 $[0, \pi/2]$ 和 $[\pi/2, \pi]$，然后对每个子区间应用梯形规则。

**梯形规则**的公式为：
$$
T(h) = \frac{h}{2} \left[ f(x_0) + 2\sum_{i=1}^{n-1} f(x_i) + f(x_n) \right]
$$
其中 $h$ 是每个子区间的步长，$x_0, x_1, \dots, x_n$ 是子区间内的节点。

对于本题，首先划分区间 $[0, \pi]$ 为两个子区间：
1. $ [0, \pi/2] $
2. $ [\pi/2, \pi] $

每个子区间的步长 $h = \frac{\pi/2 - 0}{1} = \frac{\pi}{2}$ 和 $h = \frac{\pi - \pi/2}{1} = \frac{\pi}{2}$。

对每个子区间应用梯形规则：
- 对于 $[0, \pi/2]$：
$$
T_1 = \frac{\pi/2}{2} \left[ f(0) + f(\pi/2) \right] = \frac{\pi/2}{2} \left[ \sin(0) + \sin(\pi/2) \right] = \frac{\pi/2}{2} \left[ 0 + 1 \right] = \frac{\pi}{4}
$$
- 对于 $[\pi/2, \pi]$：
$$
T_2 = \frac{\pi/2}{2} \left[ f(\pi/2) + f(\pi) \right] = \frac{\pi/2}{2} \left[ \sin(\pi/2) + \sin(\pi) \right] = \frac{\pi/2}{2} \left[ 1 + 0 \right] = \frac{\pi}{4}
$$

因此，复合梯形公式的近似值为：
$$
T_{\text{total}} = T_1 + T_2 = \frac{\pi}{4} + \frac{\pi}{4} = \frac{\pi}{2}
$$

### 方法 2: 简单 Simpson 公式

简单 Simpson 公式用于单个区间，可以用来近似计算积分。公式为：
$$
S(h) = \frac{h}{3} \left[ f(x_0) + 4f(x_1) + f(x_2) \right]
$$
其中 $x_0, x_1, x_2$ 是区间的端点和中点，$h$ 是区间的宽度。

对于本题，区间是 $[0, \pi]$，其中 $x_0 = 0$, $x_1 = \pi/2$, $x_2 = \pi$，步长 $h = \frac{\pi - 0}{2} = \frac{\pi}{2}$。

应用 Simpson 公式：
$$
S = \frac{\pi/2}{3} \left[ f(0) + 4f(\pi/2) + f(\pi) \right] = \frac{\pi/2}{3} \left[ \sin(0) + 4\sin(\pi/2) + \sin(\pi) \right]
$$
$$
S = \frac{\pi/2}{3} \left[ 0 + 4(1) + 0 \right] = \frac{\pi/2}{3} \times 4 = \frac{2\pi}{3}
$$

### 总结

- 使用复合梯形公式得到的近似值为 $\frac{\pi}{2}$
- 使用简单 Simpson 公式得到的近似值为 $\frac{2\pi}{3}$

这两种方法的结果都接近积分的确切值 $2$，但简单 Simpson 公式的结果通常会更准确。

## 2

### 问题 1: 显式格式与隐式格式离散化

已知衰变方程：
$$ \frac{dN}{dt} = -kN, \quad N(0) = N_0 $$

在时间步长$\Delta t$下，可以使用显式格式和隐式格式离散化该方程。

#### 显式格式：
显式格式是通过当前时刻的已知信息来求解下一时刻的值。在显式格式中，离散方程为：
$$ N_{n+1} = N_n - k N_n \Delta t = N_n(1 - k \Delta t) $$

其中，$N_n$表示第$n$个时刻的同位素量，$k$是衰变常数，$\Delta t$是时间步长。

#### 隐式格式：
隐式格式是通过下一时刻的未知信息来求解当前时刻的值。在隐式格式中，离散方程为：
$$ N_{n+1} = N_n - k N_{n+1} \Delta t $$

将其改写为：
$$ N_{n+1} + k N_{n+1} \Delta t = N_n $$
$$ N_{n+1}(1 + k \Delta t) = N_n $$
$$ N_{n+1} = \frac{N_n}{1 + k \Delta t} $$

### 问题 2: 计算显式格式与隐式格式的解

我们可以使用Python编程来计算显式格式和隐式格式的解。已知半衰期$T_{1/2} = 10$年，衰变常数$k$由半衰期公式计算：
$$ k = \frac{\ln 2}{T_{1/2}} = \frac{\ln 2}{10} $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 已知参数
T_half = 10  # 半衰期（年）
k = np.log(2) / T_half  # 衰变常数
N0 = 100  # 初始同位素量
time_steps = [0, 5, 10, 15, 20]  # 需要计算的时间点
dt1 = 1  # 时间步长为1年
dt2 = 5  # 时间步长为5年

# 显式格式递推
def explicit_format(N0, k, dt, time_steps):
    N = [N0]
    for t in range(1, len(time_steps)):
        N.append(N[-1] * (1 - k * dt))  # 显式格式公式
    return N

# 隐式格式递推
def implicit_format(N0, k, dt, time_steps):
    N = [N0]
    for t in range(1, len(time_steps)):
        N.append(N[-1] / (1 + k * dt))  # 隐式格式公式
    return N

# 对于dt = 1年和dt = 5年，计算显式格式和隐式格式
time_steps_1 = time_steps  # 时间步长为1年时的时间点
explicit_1 = explicit_format(N0, k, dt1, time_steps_1)
implicit_1 = implicit_format(N0, k, dt1, time_steps_1)

time_steps_2 = time_steps  # 时间步长为5年时的时间点
explicit_2 = explicit_format(N0, k, dt2, time_steps_2)
implicit_2 = implicit_format(N0, k, dt2, time_steps_2)

# 打印结果
print(f"时间步长为1年时，显式格式计算结果：{explicit_1}")
print(f"时间步长为1年时，隐式格式计算结果：{implicit_1}")
print(f"时间步长为5年时，显式格式计算结果：{explicit_2}")
print(f"时间步长为5年时，隐式格式计算结果：{implicit_2}")

# 绘制结果
plt.figure(figsize=(10, 6))

# 显式格式图
plt.subplot(2, 1, 1)
plt.plot(time_steps_1, explicit_1, label='显式格式 (dt=1)', marker='o', linestyle='-', color='blue')
plt.plot(time_steps_1, implicit_1, label='隐式格式 (dt=1)', marker='o', linestyle='--', color='green')
plt.title("Explicit vs. implicit formatting (dt=1year)")
plt.xlabel("Time (year)")
plt.ylabel("Isotopic Quantity")
plt.legend()

# 显式格式图 (时间步长为5年)
plt.subplot(2, 1, 2)
plt.plot(time_steps_2, explicit_2, label='显式格式 (dt=5)', marker='o', linestyle='-', color='blue')
plt.plot(time_steps_2, implicit_2, label='隐式格式 (dt=5)', marker='o', linestyle='--', color='green')
plt.title("Explicit vs. implicit formatting (dt=5year)")
plt.xlabel("Time (year)")
plt.ylabel("Isotopic Quantity")
plt.legend()

plt.tight_layout()
plt.show()

#### 计算结果说明：
- **显式格式**：随着时间的推移，显式格式的计算值会逐渐减小，并且在较大时间步长（如$\Delta t = 5$年）时，解会出现数值不稳定的情况，即同位素量可能会迅速减小甚至变为负值，尤其是当时间步长较大时。
- **隐式格式**：隐式格式的计算结果相对更加稳定，解随着时间的推移逐渐减小，但在大时间步长下依然可以保持稳定，不会发生数值不稳定的情况。

### 问题 3: 显式格式与隐式格式的稳定性分析

#### 隐式格式的无条件稳定性：
隐式格式具有**无条件稳定性**，即它对于任何时间步长$\Delta t$都能够保持稳定。原因在于隐式格式中，$N_{n+1}$与$N_n$的关系由$N_{n+1}$自身决定，而显式格式是基于$N_n$来推算$N_{n+1}$的。隐式格式中的分母$1 + k \Delta t$可以有效地“抵消”较大时间步长带来的误差，从而保持解的稳定。

#### 显式格式的稳定性条件：
显式格式要求时间步长$\Delta t$满足稳定性条件：
$$ \Delta t \leq \frac{2}{k} $$
该条件确保了显式格式在给定的衰变常数$k$下，解不会变得不稳定。若$\Delta t$过大，则显式格式的计算结果会变得不稳定，导致同位素量的迅速减小，甚至出现负值。

### 总结：
- **显式格式**容易受时间步长的影响，步长过大时可能导致不稳定。
- **隐式格式**稳定性更好，无论步长多大都能够保持稳定，因此适用于较大时间步长的情况。

## 3
### 半离散化公式：
根据有限差分法，空间导数的二阶导数离散为：
$$
\frac{\partial^2 u}{\partial x^2} \approx \frac{u_{i-1} - 2u_i + u_{i+1}}{h^2}.
$$
因此，对于每个离散点 \(x_i\)，得到如下常微分方程：
$$
\frac{du_i}{dt} = \alpha \frac{u_{i-1} - 2u_i + u_{i+1}}{h^2}, \quad i = 1, 2, \dots, N-1.
$$

### 显式欧拉法更新公式：
$$
u_i^{n+1} = u_i^n + \Delta t \cdot \frac{\alpha}{h^2} \left(u_{i-1}^n - 2u_i^n + u_{i+1}^n\right),
$$
其中边界条件 \(u_0 = u_N = 0\)。

---

### 初始条件：
在 \(t = 0\) 时，初始值为：
$$
u_i(0) = \sin\left(\frac{\pi x_i}{L}\right).
$$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 参数定义
L = 1.0                # 空间长度
alpha = 0.01           # 热扩散系数
N = 5                  # 空间离散分点数（包含边界点）
h = L / N              # 空间步长
dt = 0.05              # 调小时间步长以提高精度
timesteps = [0, 0.5, 1.0, 1.0]  # 计算的时间点

# 空间离散点 (去掉边界点)
x = np.linspace(h, L - h, N - 1)

# 初始条件: u(x, 0) = sin(pi * x / L)
u_initial = np.sin(np.pi * x / L)

# 计算最大时间步数
max_timestep = int(1.0 / dt)  # 改为精确步数，确保覆盖 t = 1.0

# 存储温度分布
u_history = [u_initial.copy()]

# 显式欧拉法计算温度分布
u = u_initial.copy()
for n in range(1, max_timestep + 1):
    # 保存上一时间步的值
    u_prev = u.copy()
    
    # 显式欧拉更新
    for i in range(len(u)):
        # 内部点更新公式
        if 0 < i < len(u) - 1:
            u[i] = u_prev[i] + dt * alpha / h**2 * (u_prev[i - 1] - 2 * u_prev[i] + u_prev[i + 1])
        # 边界条件
        elif i == 0 or i == len(u) - 1:
            u[i] = 0

    # 在特定时间点记录结果
    current_time = n * dt
    for t in timesteps:
        if np.isclose(current_time, t):  # 严格匹配时间点
            u_history.append(u.copy())

# 整理输出
results = {f"t={round(t, 1)}": u for t, u in zip([0] + timesteps, u_history)}

# 打印温度分布
for t, u in results.items():
    print(f"{t}: {u}")

# 可视化温度分布随时间变化
for t, u in results.items():
    plt.plot(x, u, label=f't={t}')

plt.xlabel('位置 x')
plt.ylabel('温度 u(x,t)')
plt.legend()
plt.title('温度分布随时间变化')
plt.grid(True)
plt.show()
